# Speech Psychology — Baseline Model
**روش:** TF-IDF + Ridge Regression (Multi-output)

هدف: پیش‌بینی ۱۰ ویژگی روانشناختی برای هر متن، مقدار بین ۰ تا ۴

## 1. Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.multioutput import MultiOutputRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error

import warnings
warnings.filterwarnings('ignore')

## 2. Load Data

In [ ]:
train = pd.read_csv('comments/train.csv')
test  = pd.read_csv('comments/test.csv')

print(f'Train shape: {train.shape}')
print(f'Test shape:  {test.shape}')
train.head(3)

## 3. Exploratory Data Analysis (EDA)

In [ ]:
TARGET_COLS = [
    'sense', 'honor', 'curse', 'despise', 'situation',
    'antihuman', 'roughness', 'slaughter', 'strike_support', 'depression_rate'
]

# آمار توصیفی ستون‌های هدف
train[TARGET_COLS].describe().round(2)

In [ ]:
# توزیع هر ویژگی
fig, axes = plt.subplots(2, 5, figsize=(18, 6))
axes = axes.flatten()

for i, col in enumerate(TARGET_COLS):
    axes[i].hist(train[col], bins=20, color='steelblue', edgecolor='white')
    axes[i].set_title(col)
    axes[i].set_xlabel('Score')
    axes[i].set_ylabel('Count')

plt.suptitle('Distribution of Target Columns', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# بررسی متون non-English
non_ascii = train['text'].apply(lambda x: any(ord(c) > 127 for c in str(x)))
print(f'تعداد متون non-ASCII: {non_ascii.sum()} از {len(train)}')

# نمونه
train[non_ascii]['text'].head(3).tolist()

In [ ]:
# طول متون
train['text_len'] = train['text'].fillna('').apply(len)
print(f"میانگین طول متن: {train['text_len'].mean():.0f} کاراکتر")
print(f"حداکثر: {train['text_len'].max()} | حداقل: {train['text_len'].min()}")

train['text_len'].hist(bins=50, figsize=(8,3), color='coral', edgecolor='white')
plt.title('Text Length Distribution')
plt.xlabel('Number of Characters')
plt.ylabel('Count')
plt.show()

## 4. Preprocessing

In [ ]:
# جدا کردن features و targets
X_train = train['text'].fillna('').astype(str)
y_train = train[TARGET_COLS]
X_test  = test['text'].fillna('').astype(str)

print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}')

## 5. TF-IDF Vectorization

TF-IDF هر کلمه را بر اساس فراوانی در متن و نادر بودن در کل corpus وزن‌دهی می‌کند.
- `ngram_range=(1,2)`: علاوه بر کلمات تکی، bigram هم در نظر می‌گیریم
- `sublinear_tf=True`: از log-scaling برای فراوانی استفاده می‌کنیم تا کلمات خیلی پرتکرار وزن کمتری بگیرند

In [ ]:
tfidf = TfidfVectorizer(
    max_features=30000,    # حداکثر ۳۰ هزار feature
    ngram_range=(1, 2),    # unigram + bigram
    sublinear_tf=True,     # log(tf) به جای tf
    min_df=2,              # کلماتی که حداقل ۲ بار آمده‌اند
    strip_accents='unicode',
    analyzer='word'
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf  = tfidf.transform(X_test)

print(f'TF-IDF Train matrix: {X_train_tfidf.shape}')
print(f'TF-IDF Test matrix:  {X_test_tfidf.shape}')

## 6. Model: Ridge Regression (Multi-Output)

Ridge Regression یه linear regression با L2 regularization است.
از `MultiOutputRegressor` استفاده می‌کنیم تا برای هر ۱۰ ستون جداگانه مدل بسازد.

In [ ]:
# ساخت و آموزش مدل
model = MultiOutputRegressor(
    Ridge(alpha=1.0),   # alpha: میزان regularization
    n_jobs=-1           # استفاده از همه CPU cores
)

model.fit(X_train_tfidf, y_train)
print('مدل آموزش دید!')

## 7. Evaluation — Cross Validation

In [ ]:
# محاسبه RMSE برای هر ستون با 5-fold CV
kf = KFold(n_splits=5, shuffle=True, random_state=42)
rmse_scores = {}

for col in TARGET_COLS:
    scores = cross_val_score(
        Ridge(alpha=1.0),
        X_train_tfidf,
        y_train[col],
        cv=kf,
        scoring='neg_root_mean_squared_error'
    )
    rmse_scores[col] = -scores.mean()

# نمایش نتایج
results_df = pd.DataFrame({'Column': list(rmse_scores.keys()),
                            'RMSE':   list(rmse_scores.values())})
results_df = results_df.sort_values('RMSE', ascending=False)

print(results_df.to_string(index=False))

mcrmse = results_df['RMSE'].mean()
score  = (1.5 - mcrmse) * (100/150) * 150
print(f'\nMCRMSE: {mcrmse:.4f}')
print(f'Estimated Score: {score:.2f} / 150')

In [ ]:
# نمودار RMSE هر ستون
plt.figure(figsize=(10, 4))
bars = plt.barh(results_df['Column'], results_df['RMSE'], color='steelblue', edgecolor='white')
plt.axvline(x=1.5, color='red', linestyle='--', label='Baseline threshold (1.5)')
plt.axvline(x=mcrmse, color='green', linestyle='--', label=f'MCRMSE = {mcrmse:.3f}')
plt.xlabel('RMSE')
plt.title('RMSE per Target Column (5-Fold CV)')
plt.legend()
plt.tight_layout()
plt.show()

## 8. Predict on Test Set

In [ ]:
# پیش‌بینی روی test
preds = model.predict(X_test_tfidf)

# clip: مقادیر خارج از بازه [0, 4] رو برش می‌زنیم
preds = np.clip(preds, 0, 4)

# ذخیره در DataFrame
output = pd.DataFrame(preds, columns=TARGET_COLS).round(3)

print(f'Output shape: {output.shape}')
output.head()

## 9. Save Output

In [ ]:
output.to_csv('output.csv', index=False)
print(f'output.csv ذخیره شد — {len(output)} سطر، {len(output.columns)} ستون')

## 10. Summary

| مرحله | روش |
|-------|-----|
| Vectorization | TF-IDF (unigram + bigram, 30K features) |
| Model | Ridge Regression (Multi-Output) |
| Evaluation | 5-Fold Cross Validation — MCRMSE |

**نقاط ضعف این baseline:**
- معنای جمله رو درک نمی‌کند (فقط کلمات رو می‌بیند)
- متون غیرانگلیسی رو به درستی handle نمی‌کند
- ترتیب کلمات برایش مهم نیست

**مرحله بعد:** استفاده از BERT embeddings برای درک عمیق‌تر متن